# 01 - Run the Flow Matching + pruning pipeline (Colab)

Runs `eval/run_flow_pruning.py` over all four LUCAS dataset variants and writes the
per-iteration adversarial AUCs that Table I and Fig. 2 of the letter are built from.

**Prerequisites**

1. A Google Drive folder holding the prepared datasets, produced once by
   `prepare_datasets.py` (see the README for how to obtain LUCAS 2015 from ESDAC).
   Point `DRIVE_ROOT` in the Configuration cell at your own folder, laid out as:

   ```
   <DRIVE_ROOT>/
   |-- prepared_datasets/
   |   |-- chem_nospectral/chem_nospectral.csv
   |   |-- chem_spectral/chem_spectral.csv
   |   |-- chem_phys_nospectral/chem_phys_nospectral.csv
   |   `-- chem_phys_spectral/chem_phys_spectral.csv
   `-- <RUN_NAME>/                <- created by this notebook
   ```
2. A GPU runtime (the letter used an NVIDIA L4).

**Produces** `results/flow_matching/<variant>/phase*.csv`, plus per-iteration sample
dumps under `saved_samples_iter_*/`, written straight to Drive so a disconnect does
not lose finished iterations.

**Runtime** the training phase alone measures roughly 55 / 57 / 12 / 13 s per
iteration for the four variants; generation and the XGBoost AUC over 11 thresholds
add to that. Budget generously and lower `RUNS` if you only want a smoke test.

In [ ]:
# ==========================================
# Configuration
# ==========================================
RUNS = 100                      # Monte Carlo iterations (the letter used 100)
REPO_URL = "https://github.com/abstratovcm/pruned-soil-fm.git"
REPO_DIR = "pruned-soil-fm"
DRIVE_ROOT = "/content/drive/MyDrive/pruned-soil-fm"   # your own folder
RUN_NAME = "ours_run1"          # fresh name per run; an existing one is overwritten
# ==========================================
print(f"Configured for {RUNS} iterations -> Drive folder: {RUN_NAME}")

In [ ]:
!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Link the checkout to Drive: read-only inputs, writable outputs.
repo_path = f"/content/{REPO_DIR}"
run_path = f"{DRIVE_ROOT}/{RUN_NAME}"

# 1. Prepared datasets (shared, read-only)
!mkdir -p "{repo_path}/data"
!rm -rf "{repo_path}/data/prepared_datasets"
!ln -s "{DRIVE_ROOT}/prepared_datasets" "{repo_path}/data/prepared_datasets"

# 2. Point results/flow_matching at Drive
!mkdir -p "{run_path}/flow_matching"
!mkdir -p "{repo_path}/results"
!rm -rf "{repo_path}/results/flow_matching"
!ln -s "{run_path}/flow_matching" "{repo_path}/results/flow_matching"

!ls -la "{repo_path}/results"

In [ ]:
# The only dependency Colab does not already ship.
!pip install -q torchdiffeq==0.2.5
!pip freeze > "{run_path}/environment.txt"

In [ ]:
%cd {repo_path}
!PYTHONPATH=. python eval/run_flow_pruning.py --N {RUNS}